# Semantic Benchmark Python Dataset Analysis

Reads artifacts from `01_extract_data.py`, summarizes counts, and renders positive semantic clone code pairs like the BCB dataset notebook.

In [ ]:
from pathlib import Path
import html
import json
import pickle
import random
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.evaluation.notebook_helpers import semantic_spec

LANGUAGE = "python"
DISPLAY_LANGUAGE = "Python"
GRAPH_TYPES = ['ast', 'cfg', 'ddg', 'cpg']
PRIMARY_GRAPH_TYPE = "cpg"

spec = semantic_spec(LANGUAGE)
DATA_DIR = spec.data_dir
OUTPUT_ROOT = spec.output_root
REPORTS_DIR = OUTPUT_ROOT / "reports"
GRAPH_MANIFEST_PATH = OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
SPECTRAL_MANIFEST_PATH = OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"

print("Project root:", PROJECT_ROOT)
print("Prepared data:", DATA_DIR)
print("Output root:", OUTPUT_ROOT)


In [ ]:
def load_code_for_ids(path: Path, wanted_ids: set[str]) -> dict[str, str]:
    code_map = {}
    if not path.exists():
        return code_map
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            method_id = str(row["idx"])
            if method_id in wanted_ids:
                code_map[method_id] = row.get("func", "")
                if len(code_map) == len(wanted_ids):
                    break
    return code_map


def load_graph_manifest() -> dict:
    if not GRAPH_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Graph manifest not found: {GRAPH_MANIFEST_PATH}. Run ../run_pipeline/02_extract_graphs.py first.")
    return json.loads(GRAPH_MANIFEST_PATH.read_text(encoding="utf-8"))


def load_graphs_for_ids(manifest: dict, wanted_ids: set[str]) -> dict[str, dict[str, nx.DiGraph]]:
    graphs = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(graphs)):
            if method_id in shard:
                graphs[method_id] = shard[method_id]
        if len(graphs) == len(wanted_ids):
            break
    return graphs


def load_spectral_features_for_ids(manifest_path: Path, wanted_ids: set[str]) -> dict[str, dict]:
    if not manifest_path.exists():
        return {}
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    features = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(features)):
            if method_id in shard:
                features[method_id] = shard[method_id]
        if len(features) == len(wanted_ids):
            break
    return features


def draw_graph(graph: nx.DiGraph | None, title: str, ax, max_nodes: int = 100) -> None:
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if graph is None or graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "empty", ha="center", va="center")
        return
    graph = nx.DiGraph(graph)
    shown = graph if graph.number_of_nodes() <= max_nodes else graph.subgraph(list(graph.nodes())[:max_nodes]).copy()
    pos = nx.spring_layout(shown, seed=42)
    nx.draw_networkx_edges(shown, pos, ax=ax, arrows=False, alpha=0.25, width=0.8)
    nx.draw_networkx_nodes(shown, pos, ax=ax, node_size=22, alpha=0.85)
    ax.text(0.01, 0.02, f"{graph.number_of_nodes()} nodes / {graph.number_of_edges()} edges", transform=ax.transAxes, fontsize=7)


def eigenvalues_for(spectral_map: dict, method_id: str, graph_type: str) -> np.ndarray:
    values = spectral_map.get(str(method_id), {}).get(graph_type, {}).get("eigenvalues", [])
    values = np.asarray(values, dtype=np.float64)
    return values[np.isfinite(values)]


def draw_eigenvalues(spectral_map: dict, method_id: str, graph_type: str, ax) -> None:
    values = eigenvalues_for(spectral_map, method_id, graph_type)
    ax.set_title(f"{graph_type.upper()} eigenvalues", fontsize=9)
    if values.size == 0:
        ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        return
    ax.plot(np.arange(1, values.size + 1), np.sort(values), linewidth=1.2)
    ax.grid(alpha=0.2)
    ax.tick_params(labelsize=8)


def render_code(method_id: str, code_map: dict[str, str], title: str) -> None:
    display(HTML(f"""
    <div style="border:1px solid #d0d7de;border-radius:6px;overflow:hidden;margin:10px 0;">
      <div style="padding:6px 10px;background:#f6f8fa;font-family:system-ui,sans-serif;font-size:13px;">{html.escape(title)} - id {html.escape(str(method_id))}</div>
      <pre style="margin:0;padding:12px;overflow-x:auto;white-space:pre-wrap;font-size:12px;line-height:1.35;max-height:460px;">{html.escape(code_map.get(str(method_id), ""))}</pre>
    </div>
    """))


def render_code_pair(left_id: str, right_id: str, code_map: dict[str, str], title: str) -> None:
    display(HTML(f"""
    <h3>{html.escape(title)}</h3>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;align-items:start;">
      <div><b>left id: {html.escape(str(left_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:460px;overflow:auto;">{html.escape(code_map.get(str(left_id), ""))}</pre></div>
      <div><b>right id: {html.escape(str(right_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:460px;overflow:auto;">{html.escape(code_map.get(str(right_id), ""))}</pre></div>
    </div>
    """))


In [ ]:
metadata_path = DATA_DIR / "metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError(f"metadata.json not found in {DATA_DIR}. Run 01_extract_data.py first.")
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))


def count_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("rb") as f:
        return sum(1 for _ in f)


def unique_code_ids_in_pairs(path: Path) -> int:
    ids = set()
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                ids.add(parts[0])
                ids.add(parts[1])
    return len(ids)


summary_df = pd.DataFrame([
    {"metric": "Prepared functions written", "count": count_lines(DATA_DIR / "data.jsonl"), "source": "data.jsonl"},
    {"metric": "Train pairs", "count": count_lines(DATA_DIR / "train.txt"), "source": "train.txt"},
    {"metric": "Unique code ids in train pairs", "count": unique_code_ids_in_pairs(DATA_DIR / "train.txt"), "source": "train.txt"},
    {"metric": "Metadata total pairs", "count": metadata.get("pairs", metadata.get("total_pairs")), "source": "metadata.json"},
    {"metric": "Metadata positive pairs", "count": metadata.get("positive_pairs", metadata.get("positive_clones")), "source": "metadata.json"},
    {"metric": "Metadata negative pairs", "count": metadata.get("negative_pairs", metadata.get("non_clones")), "source": "metadata.json"},
])
display(summary_df)
display(pd.DataFrame([metadata]).T.rename(columns={0: "value"}).head(100))


In [ ]:
def reservoir_sample_by_label(path: Path, label: int, n: int, seed: int) -> list[tuple[str, str, int]]:
    rng = random.Random(seed)
    sample = []
    seen = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            left, right, raw_label = line.rstrip("\n").split("\t")[:3]
            raw_label = int(raw_label)
            if raw_label != label:
                continue
            seen += 1
            row = (left, right, raw_label)
            if len(sample) < n:
                sample.append(row)
            else:
                slot = rng.randrange(seen)
                if slot < n:
                    sample[slot] = row
    return sample


label_counts = pd.read_csv(DATA_DIR / "train.txt", sep="\t", header=None, names=["left_id", "right_id", "label"])["label"].value_counts().sort_index()
display(label_counts.rename_axis("label").to_frame("pairs"))

positive_pairs = reservoir_sample_by_label(DATA_DIR / "train.txt", label=1, n=4, seed=42)
wanted_ids = {item for pair in positive_pairs for item in pair[:2]}
code_map = load_code_for_ids(DATA_DIR / "data.jsonl", wanted_ids)
positive_pairs = [pair for pair in positive_pairs if pair[0] in code_map and pair[1] in code_map]

print("Loaded code snippets for examples:", len(code_map))
print("Positive clone examples:", len(positive_pairs))


In [ ]:
for idx, pair in enumerate(positive_pairs, start=1):
    render_code_pair(pair[0], pair[1], code_map, f"Positive semantic clone example {idx}")


In [ ]:
def stage_runtime_row(stage: str) -> dict:
    timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}
    record = timings.get("stages", {}).get(stage, {})
    seconds = record.get("seconds")
    minutes = record.get("minutes")
    if minutes is None and isinstance(seconds, (int, float)):
        minutes = seconds / 60
    return {
        "stage": stage,
        "seconds": round(seconds, 2) if isinstance(seconds, (int, float)) else None,
        "minutes": round(minutes, 2) if isinstance(minutes, (int, float)) else None,
        "updated_at_utc": record.get("updated_at_utc"),
        "status": "recorded" if isinstance(seconds, (int, float)) else "not recorded yet",
        "source": str(PIPELINE_TIMINGS_PATH),
    }


display(pd.DataFrame([stage_runtime_row("01_extract_data")]))